## Setting up

In [2]:
%%capture
%pip install -U bitsandbytes
%pip install -U transformers
%pip install -U accelerate
%pip install -U peft
%pip install -U trl

In [4]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [7]:
import warnings
warnings.filterwarnings("ignore")

**Import packages**

In [9]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
# from peft import LoraConfig, PeftConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer
from trl import SFTTrainer, SFTConfig
#from trl import setup_chat_format
from transformers import (AutoModelForCausalLM, 
                          AutoTokenizer, 
                          BitsAndBytesConfig, 
                          TrainingArguments, 
                          pipeline, 
                          logging)
from sklearn.metrics import (accuracy_score, 
                             classification_report, 
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
print(f"pytorch version {torch.__version__}")

pytorch version 2.11.0+cu128


In [11]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

working on cuda:0


In [8]:
# torch.backends.cuda.enable_mem_efficient_sdp(False)
# torch.backends.cuda.enable_flash_sdp(False)

**Load the data**

In [12]:
df = pd.read_csv("/kaggle/input/datasets/forouzanrezaei/sentiment-analysis-for-mental-health/Combined Data.csv",index_col = "Unnamed: 0")
df.loc[:,'status'] = df.loc[:,'status'].str.replace('Bi-Polar','Bipolar')
df = df[(df.status != "Personality disorder") & (df.status != "Stress") & (df.status != "Suicidal")]
df.head()

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38520 entries, 0 to 53042
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   statement  38365 non-null  object
 1   status     38520 non-null  object
dtypes: object(2)
memory usage: 902.8+ KB


In [14]:
df.isnull().sum()

statement    155
status         0
dtype: int64

In [16]:
df.isnull().sum().sum()

np.int64(155)

In [17]:
df[df['statement'].isnull()].head(1)

,statement,status
293,NaN,Anxiety


In [18]:
df = df.dropna(subset=['statement'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38365 entries, 0 to 53042
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   statement  38365 non-null  object
 1   status     38365 non-null  object
dtypes: object(2)
memory usage: 899.2+ KB


In [19]:
df.isnull().sum()

statement    0
status       0
dtype: int64

In [20]:
df = df.reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38365 entries, 0 to 38364
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   statement  38365 non-null  object
 1   status     38365 non-null  object
dtypes: object(2)
memory usage: 599.6+ KB


In [21]:
# Shuffle the DataFrame and select only 3000 rows
df = df.sample(frac=1, random_state=85).reset_index(drop=True).head(3000)

# Split the DataFrame
train_size = 0.8
eval_size = 0.1

# Calculate sizes
train_end = int(train_size * len(df))
eval_end = train_end + int(eval_size * len(df))

# Split the data
X_train = df[:train_end]
X_eval = df[train_end:eval_end]
X_test = df[eval_end:]

# # Define the prompt generation functions
# def generate_prompt(data_point):
#     return f"""
#             Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
# text: {data_point["statement"]}
# label: {data_point["status"]}""".strip()

EOS = "<|im_end|>"

def generate_prompt(data_point):
    return f"""
            Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
text: {data_point["statement"]}
label: {data_point["status"]}{EOS}""".strip()

def generate_test_prompt(data_point):
    return f"""
            Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.
text: {data_point["statement"]}
label: """.strip()

# Generate prompts for training and evaluation data
X_train.loc[:,'text'] = X_train.apply(generate_prompt, axis=1)
X_eval.loc[:,'text'] = X_eval.apply(generate_prompt, axis=1)

# Generate test prompts and extract true labels
y_true = X_test.loc[:,'status']
X_test = pd.DataFrame(X_test.apply(generate_test_prompt, axis=1), columns=["text"])



In [22]:
X_train.status.value_counts()

status
Normal        1020
Depression     969
Anxiety        246
Bipolar        165
Name: count, dtype: int64

In [23]:
y_true.value_counts()

status
Depression    130
Normal        116
Anxiety        31
Bipolar        23
Name: count, dtype: int64

In [26]:
# Convert to datasets
train_data = Dataset.from_pandas(X_train[["text"]])
eval_data = Dataset.from_pandas(X_eval[["text"]])

In [27]:
train_data['text'][3]

'Classify the text into Normal, Depression, Anxiety, Bipolar, and return the answer as the corresponding mental health disorder label.\ntext: I have always wanted to had had urges to but I have never actually tried. I want to so bad but I am stuck here because other people in my family died and my little brother is still just a kid. I am hurting so bad but my brain says only people who try to kill themself or have worse disorders should go. If I try to I really 100% think Ill die and I wish I had an easy moral out so bad. I feel so fake because I have never tried to kill myself.\nlabel: Depression<|im_end|>'

## Loading the model and tokenizer

In [28]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype="float16",
    quantization_config=bnb_config, 
)

model.config.use_cache = False
# model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [29]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Model evalution before fine-tuning

In [27]:
# def predict(test, model, tokenizer):
#     y_pred = []
#     categories = ["Normal", "Depression", "Anxiety", "Bipolar"]
    
#     for i in tqdm(range(len(test))):
#         prompt = test.iloc[i]["text"]
#         pipe = pipeline(task="text-generation", 
#                         model=model, 
#                         tokenizer=tokenizer, 
#                         max_new_tokens=6, 
#                         temperature=0.1)
        
#         result = pipe(prompt)
#         answer = result[0]['generated_text'].split("label:")[-1].strip()
        
#         # Determine the predicted category
#         for category in categories:
#             if category.lower() in answer.lower():
#                 y_pred.append(category)
#                 break
#         else:
#             y_pred.append("none")
    
#     return y_pred

In [30]:
def predict(test, model, tokenizer):
    y_pred = []
    categories = ["Normal", "Depression", "Anxiety", "Bipolar"]

    pipe = pipeline(task="text-generation",
                    model=model,
                    tokenizer=tokenizer,
                    max_new_tokens=6,
                    do_sample=False,
                    return_full_text=False)

    for i in tqdm(range(len(test))):
        result = pipe(test.iloc[i]["text"])
        answer = result[0]["generated_text"].strip()

        for category in categories:
            if category.lower() in answer.lower():
                y_pred.append(category)
                break
        else:
            y_pred.append("none")

    return y_pred

In [31]:
y_pred = predict(X_test, model, tokenizer)

100%|██████████| 300/300 [02:10<00:00,  2.30it/s]


In [32]:
def evaluate(y_true, y_pred):
    labels = ["Normal", "Depression", "Anxiety", "Bipolar"]
    mapping = {label: idx for idx, label in enumerate(labels)}
    
    def map_func(x):
        return mapping.get(x, -1)  # Map to -1 if not found, but should not occur with correct data
    
    y_true_mapped = np.vectorize(map_func)(y_true)
    y_pred_mapped = np.vectorize(map_func)(y_pred)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true_mapped, y_pred=y_pred_mapped)
    print(f'Accuracy: {accuracy:.3f}')
    
    # Generate accuracy report
    unique_labels = set(y_true_mapped)  # Get unique labels
    
    for label in unique_labels:
        label_indices = [i for i in range(len(y_true_mapped)) if y_true_mapped[i] == label]
        label_y_true = [y_true_mapped[i] for i in label_indices]
        label_y_pred = [y_pred_mapped[i] for i in label_indices]
        label_accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f'Accuracy for label {labels[label]}: {label_accuracy:.3f}')
        
    # Generate classification report
    class_report = classification_report(y_true=y_true_mapped, y_pred=y_pred_mapped, target_names=labels, labels=list(range(len(labels))))
    print('\nClassification Report:')
    print(class_report)
    
    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true_mapped, y_pred=y_pred_mapped, labels=list(range(len(labels))))
    print('\nConfusion Matrix:')
    print(conf_matrix)

In [34]:
evaluate(y_true, y_pred)

Accuracy: 0.373
Accuracy for label Normal: 0.069
Accuracy for label Depression: 0.515
Accuracy for label Anxiety: 0.903
Accuracy for label Bipolar: 0.391

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.07      0.13       116
  Depression       0.62      0.52      0.56       130
     Anxiety       0.16      0.90      0.27        31
     Bipolar       0.82      0.39      0.53        23

    accuracy                           0.37       300
   macro avg       0.65      0.47      0.37       300
weighted avg       0.73      0.37      0.36       300


Confusion Matrix:
[[ 8 33 75  0]
 [ 0 67 61  2]
 [ 0  3 28  0]
 [ 0  5  9  9]]


## Extracting the linear modules names

In [35]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

In [36]:
modules = find_all_linear_names(model)
modules

['o_proj', 'v_proj', 'k_proj', 'down_proj', 'up_proj', 'gate_proj', 'q_proj']

In [37]:
lengths = [len(tokenizer(t)["input_ids"]) for t in train_data["text"]]
print("max:", max(lengths), "| mean:", int(np.mean(lengths)), "| 95th pct:", int(np.percentile(lengths, 95)))

max: 5813 | mean: 154 | 95th pct: 458


## Setting up the model

In [40]:
from trl import SFTTrainer, SFTConfig

output_dir="Qwen2.5-fine-tuned-model"

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules,
)

training_arguments = SFTConfig(
    output_dir=output_dir,                    # directory to save and repository id
    num_train_epochs=1,                       # number of training epochs
    per_device_train_batch_size=1,            # batch size per device during training
    gradient_accumulation_steps=8,            # number of steps before performing a backward/update pass
    gradient_checkpointing=True,              # use gradient checkpointing to save memory
    optim="paged_adamw_32bit",
    logging_steps=1,                         
    learning_rate=2e-4,                       # learning rate, based on QLoRA paper
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,                        # max gradient norm based on QLoRA paper
    max_steps=-1,
    warmup_steps=0.03,                       # warmup ratio based on QLoRA paper
    # group_by_length=False,
    lr_scheduler_type="cosine",               # use cosine learning rate scheduler
    report_to="none",                  # report metrics to w&b
    eval_strategy="steps",              # save checkpoint every epoch
    eval_steps = 0.2,
    dataset_text_field="text",
    max_length=None,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,

)

Adding EOS to train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

In [41]:
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

## Model Training

In [42]:
# Train model
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
60,1.961254,2.366343,2.375681,0.521042,67996.000000
120,2.188656,2.349704,2.332475,0.521341,144881.000000
180,2.008976,2.339922,2.347616,0.522706,221215.000000
240,2.193571,2.334805,2.350817,0.523540,295505.000000
300,2.201322,2.333942,2.336052,0.523324,371790.000000


TrainOutput(global_step=300, training_loss=2.390962908665339, metrics={'train_runtime': 1692.4238, 'train_samples_per_second': 1.418, 'train_steps_per_second': 0.177, 'total_flos': 3087789594163200.0, 'train_loss': 2.390962908665339, 'epoch': 1.0})

In [43]:
model.config.use_cache = True

## Saving the model and tokenizer

In [44]:
# Save trained model and tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

('Qwen2.5-fine-tuned-model/tokenizer_config.json',
 'Qwen2.5-fine-tuned-model/chat_template.jinja',
 'Qwen2.5-fine-tuned-model/tokenizer.json')

## Testing model after fine-tuning

In [45]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

100%|██████████| 300/300 [01:28<00:00,  3.40it/s]

Accuracy: 0.890
Accuracy for label Normal: 0.922
Accuracy for label Depression: 0.946
Accuracy for label Anxiety: 0.581
Accuracy for label Bipolar: 0.826

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.92      0.93       116
  Depression       0.88      0.95      0.91       130
     Anxiety       0.72      0.58      0.64        31
     Bipolar       0.86      0.83      0.84        23

   micro avg       0.89      0.89      0.89       300
   macro avg       0.85      0.82      0.83       300
weighted avg       0.89      0.89      0.89       300


Confusion Matrix:
[[107   6   3   0]
 [  1 123   4   2]
 [  4   7  18   1]
 [  1   3   0  19]]
